In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("API key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

API key loaded: True


In [2]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://gnits.ac.in")
documents = loader.load()

print("Documents loaded:", len(documents))
print(documents[0].page_content[:1000])

C:\Users\NAWAB\AppData\Local\Temp\ipykernel_4948\3053625840.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Documents loaded: 1









G. Narayanamma Institute of Technology and Sciences - GNITS














































































 














	Skip to content








About

About
Legacy
Leadership & Governance
Strategic Plan
Organogram
Academic & Administrative Committees
Awards & Recognitions
Accreditations
Why GNITS
Governance

Governing Council Members
College Academic Council


Policies
Annual Reports
Audit Reports

Welfare Measures


Mandatory Disclosures


Academics

Bachelors Programs

Computer Science Engineering
Information Technology
Computer Science & Engineering (Artificial Intelligence & Machine Learning)
Computer Science – Data science
Computer Science and Technology
Electronics and Communication Engineering
Electrical and Electronics Engineering
Electronics and Telematics Engineering


Auxilary Departments

Humanities and Mathematics
Basic Sciences Department
Mechanical
Civil
Administration


Syllabus ( UG & PG )
Masters (PG) Pro

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0].page_content)

Number of chunks: 27

First chunk:
G. Narayanamma Institute of Technology and Sciences - GNITS














































































 














	Skip to content








About

About
Legacy
Leadership & Governance
Strategic Plan
Organogram
Academic & Administrative Committees
Awards & Recognitions
Accreditations
Why GNITS
Governance

Governing Council Members
College Academic Council


Policies
Annual Reports
Audit Reports

Welfare Measures


Mandatory Disclosures


Academics

Bachelors Programs

Computer Science Engineering
Information Technology
Computer Science & Engineering (Artificial Intelligence & Machine Learning)
Computer Science – Data science
Computer Science and Technology
Electronics and Communication Engineering
Electrical and Electronics Engineering
Electronics and Telematics Engineering


Auxilary Departments

Humanities and Mathematics
Basic Sciences Department
Mechanical
Civil
Administration


Syllabus ( UG & PG )
Masters (P

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vector = embeddings.embed_query(chunks[0].page_content)

print("Embedding created successfully!")
print("Vector length:", len(vector))
print("First 5 values:", vector[:5])

Embedding created successfully!
Vector length: 3072
First 5 values: [0.0073897783, 0.01108736, 0.007369884, -0.02610635, 0.0062913997]


In [5]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="gnits_collection",
    embedding_function=embeddings
)

print("Chroma vector store created successfully!")

Chroma vector store created successfully!


In [7]:
collection = vector_store._collection

print("Chroma collection ready!")

Chroma collection ready!


In [8]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [
    chunk.page_content.lower().split()
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 retriever created successfully!")
print("Documents indexed:", len(tokenized_chunks))

BM25 retriever created successfully!
Documents indexed: 27


In [9]:
chunk_embeddings = []

for i, chunk in enumerate(chunks):
    vector = embeddings.embed_query(chunk.page_content)
    chunk_embeddings.append(vector)

print("Embeddings created:", len(chunk_embeddings))
print("Vector dimension:", len(chunk_embeddings[0]))

Embeddings created: 27
Vector dimension: 3072


In [10]:
import numpy as np

chat_history = []

print("Conversation memory initialized!")

Conversation memory initialized!


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    max_retries=2,
    max_output_tokens=1024
)

print("LLM ready!")

LLM ready!


In [12]:
def rewrite_query(question, chat_history):

    if not chat_history:
        return question

    lower_question = question.lower()

    follow_up_phrases = [
        "what about",
        "how about",
        "tell me more",
        "and what about",
        "what else"
    ]

    is_follow_up = any(
        phrase in lower_question
        for phrase in follow_up_phrases
    )

    if is_follow_up:
        previous_question = chat_history[-1]["question"]
        return f"{previous_question} {question}"

    return question

In [13]:
def campus_qa(question, chat_history):

    # Query rewriting
    rewritten_query = rewrite_query(
        question,
        chat_history
    )

    # Vector search
    query_embedding = embeddings.embed_query(
        rewritten_query
    )

    vector_scores = []

    for vector in chunk_embeddings:

        similarity = np.dot(
            query_embedding,
            vector
        ) / (
            np.linalg.norm(query_embedding)
            * np.linalg.norm(vector)
        )

        vector_scores.append(similarity)

    vector_scores = np.array(vector_scores)

    # BM25 keyword search
    tokenized_query = rewritten_query.lower().split()

    bm25_scores = bm25.get_scores(
        tokenized_query
    )

    bm25_scores = np.array(bm25_scores)

    # Normalize BM25 scores
    if bm25_scores.max() > 0:
        bm25_normalized = (
            bm25_scores /
            (bm25_scores.max() + 1e-8)
        )
    else:
        bm25_normalized = bm25_scores

    # Hybrid retrieval
    hybrid_scores = (
        0.5 * bm25_normalized +
        0.5 * vector_scores
    )

    # Top-K retrieval
    top_k = 5

    top_k_indices = np.argsort(
        hybrid_scores
    )[-top_k:][::-1]

    # Reranking
    reranked_indices = sorted(
        top_k_indices,
        key=lambda i: hybrid_scores[i],
        reverse=True
    )

    # Top-N
    top_n = 3

    final_indices = reranked_indices[:top_n]

    # Build context
    context_parts = []

    for i in final_indices:
        context_parts.append(
            chunks[i].page_content
        )

    context = "\n\n".join(context_parts)

    # Sources
    sources = []

    for i in final_indices:

        source = chunks[i].metadata.get(
            "source",
            "GNITS official website"
        )

        if source not in sources:
            sources.append(source)

    # RAG prompt
    prompt = f"""
You are a helpful GNITS university assistant.

Answer the user's question using ONLY the provided context.

Do not use outside knowledge.

If the answer is not available in the context, say:

"I could not find this information in the provided GNITS information."

Conversation history:
{chat_history}

Retrieved context:
{context}

User question:
{question}

Give a clear and concise answer.
"""

    # LLM generation
    response = llm.invoke(prompt)

    # Clean Gemini response
    if isinstance(response.content, list):

        answer_parts = []

        for item in response.content:

            if isinstance(item, dict):

                text = item.get("text", "")

                if text:
                    answer_parts.append(text)

            elif isinstance(item, str):

                answer_parts.append(item)

        answer = "".join(answer_parts)

    else:
        answer = str(response.content)

    # Store conversation
    chat_history.append({
        "question": question,
        "answer": answer
    })

    return answer, sources, rewritten_query

In [14]:
question = "What courses are offered by GNITS?"

answer, sources, rewritten_query = campus_qa(
    question,
    chat_history
)

print("Question:")
print(question)

print("\nRewritten Query:")
print(rewritten_query)

print("\nAnswer:")
print(answer)

print("\nSources:")
for source in sources:
    print(source)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Question:
What courses are offered by GNITS?

Rewritten Query:
What courses are offered by GNITS?

Answer:
Based on the provided context, the following courses/departments are listed:

* EEE
* CSE
* ECE
* IT
* CSM & CSD
* ETM

Sources:
https://gnits.ac.in


In [15]:
question = "What about the engineering courses?"

answer, sources, rewritten_query = campus_qa(
    question,
    chat_history
)

print("Question:")
print(question)

print("\nRewritten Query:")
print(rewritten_query)

print("\nAnswer:")
print(answer)

print("\nSources:")
for source in sources:
    print(source)

Question:
What about the engineering courses?

Rewritten Query:
What courses are offered by GNITS? What about the engineering courses?

Answer:
Based on the provided context, the engineering programs/courses listed at GNITS are:

* **EEE**
* **CSE**
* **ECE**
* **IT**
* **CSM & CSD**
* **ETM**

These programs are designed to equip students with the skills and knowledge needed to excel in today’s rapidly changing technological landscape, supported by mentorship from distinguished faculty combining academic rigor with real-world expertise.

Sources:
https://gnits.ac.in


In [16]:
question = "Which of them are related to computer science?"

answer, sources, rewritten_query = campus_qa(
    question,
    chat_history
)

print("Question:")
print(question)

print("\nRewritten Query:")
print(rewritten_query)

print("\nAnswer:")
print(answer)

print("\nSources:")
for source in sources:
    print(source)

Question:
Which of them are related to computer science?

Rewritten Query:
Which of them are related to computer science?

Answer:
Based on the provided context, the programs related to computer science are:

* **Computer Science Engineering**
* **Computer Science & Engineering (Artificial Intelligence & Machine Learning)**
* **Computer Science – Data Science** (or **Computer Science & Engineering (Data Science)**)
* **Computer Science and Technology** 

*(Note: **Information Technology** is also listed among the related academic degree programs).*

Sources:
https://gnits.ac.in


In [17]:
print("Conversation Memory:\n")

for i, chat in enumerate(chat_history, start=1):

    print(f"Conversation {i}")
    print("Question:", chat["question"])
    print("Answer:", chat["answer"])
    print("-" * 60)

Conversation Memory:

Conversation 1
Question: What courses are offered by GNITS?
Answer: Based on the provided context, the following courses/departments are listed:

* EEE
* CSE
* ECE
* IT
* CSM & CSD
* ETM
------------------------------------------------------------
Conversation 2
Question: What about the engineering courses?
Answer: Based on the provided context, the engineering programs/courses listed at GNITS are:

* **EEE**
* **CSE**
* **ECE**
* **IT**
* **CSM & CSD**
* **ETM**

These programs are designed to equip students with the skills and knowledge needed to excel in today’s rapidly changing technological landscape, supported by mentorship from distinguished faculty combining academic rigor with real-world expertise.
------------------------------------------------------------
Conversation 3
Question: Which of them are related to computer science?
Answer: Based on the provided context, the programs related to computer science are:

* **Computer Science Engineering**
* **Com

In [18]:
print("""
CAMPUS Q&A ASSISTANT

University Website / PDFs
        ↓
Document Loaders
        ↓
Documents
        ↓
Chunking
        ↓
Embeddings
        ↓
Chroma Vector Store
        ↓
BM25 Keyword Index
        ↓
User Question
        ↓
Conversation Memory
        ↓
Query Understanding / Rewriting
        ↓
 ┌───────────────────────┐
 ↓                       ↓
Vector Search        BM25 Search
 ↓                       ↓
 └───────────┬───────────┘
             ↓
       Hybrid Retrieval
             ↓
        Score Fusion
             ↓
           Top-K
             ↓
         Reranking
             ↓
           Top-N
             ↓
          Context
             ↓
            LLM
             ↓
        Final Answer
             ↓
       Source Information
""")


CAMPUS Q&A ASSISTANT

University Website / PDFs
        ↓
Document Loaders
        ↓
Documents
        ↓
Chunking
        ↓
Embeddings
        ↓
Chroma Vector Store
        ↓
BM25 Keyword Index
        ↓
User Question
        ↓
Conversation Memory
        ↓
Query Understanding / Rewriting
        ↓
 ┌───────────────────────┐
 ↓                       ↓
Vector Search        BM25 Search
 ↓                       ↓
 └───────────┬───────────┘
             ↓
       Hybrid Retrieval
             ↓
        Score Fusion
             ↓
           Top-K
             ↓
         Reranking
             ↓
           Top-N
             ↓
          Context
             ↓
            LLM
             ↓
        Final Answer
             ↓
       Source Information

